# Imports

In [82]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from typing import Tuple

# Load train.bin file

In [2]:
binary_file_path = '../data/tinystories/processed/train.bin'

In [3]:
np_arr = np.memmap(
    filename=binary_file_path,
    dtype=np.uint16,
    mode='r'
)

In [4]:
np_arr

memmap([3198, 1110,   11, ..., 8030, 3290, 3706],
       shape=(471872517,), dtype=uint16)

In [5]:
tot_tokens = len(np_arr)
B = 32  # batch_size
T = 128 # ctx_len

num_batches = tot_tokens//(B*T)
print(tot_tokens, B, T)
print(num_batches)


471872517 32 128
115203


In [6]:
471872517 - (115203 * (B*T))

1029

In [7]:
int(np.floor(471872517/(B*T)))

115203

# Dataset: 

Always make sure we have 1 extra token after slotting in for n*ctx_length where n is an integer [1, ]

In [78]:
class TokenDataset(Dataset):

    def __init__(self, binary_file_path:str, dtype:np.dtype, T:int, debug_mode=False):
        T = max(T, 4)
        if debug_mode:
            tokens = np.arange(1, 101)
        else:
            tokens = np.memmap(
                filename=binary_file_path,
                dtype=dtype,
                mode='r'
            )
        
        # Validate and reject the last few tokens
        if len(tokens) <= T :
            raise ValueError(f"Current value of sequence length {T} too large for dataset, please reduce T.")
        elif len(tokens)%T == 0:
            tokens = tokens[:-(T -1)]
        elif len(tokens)%T == 1:
            tokens = tokens[:]
        elif len(tokens)%T > 1:
            tokens = tokens[:-(len(tokens)%T-1)]

        self.T = T
        self.tokens = tokens


    def __len__(self) -> int:
        return len(self.tokens)//self.T


    def __getitem__(self, idx:int) -> torch.tensor:
        buffer = self.tokens[idx*self.T:((idx+1)*self.T)+1]
        x = torch.tensor(buffer[:-1])
        y = torch.tensor(buffer[1:])

        return x, y

binary_file_path = '../data/tinystories/processed/train.bin'
T = -2 #49 #50 #99 #100
tok_ds = TokenDataset(binary_file_path, np.uint16, T, True)
print(tok_ds.tokens)
print('-'*100)
print(len(tok_ds))

[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48
 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72
 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96
 97]
----------------------------------------------------------------------------------------------------
24


In [76]:
# Much Simpler Logic

class TokenDataset_v2(Dataset):

    def __init__(self, binary_file_path:str, dtype:np.dtype, T:int, debug_mode=False):
        T = max(T, 4)
        if debug_mode:
            tokens = np.arange(1, 101)
        else:
            tokens = np.memmap(
                filename=binary_file_path,
                dtype=dtype,
                mode='r'
            )
        
        # Validate and reject the last few tokens
        if len(tokens) <= T :
            raise ValueError(f"Current value of sequence length {T} too large for dataset, please reduce T.")

        self.T = T
        self.tokens = tokens


    def __len__(self) -> int:
        return (len(self.tokens)-1)//self.T


    def __getitem__(self, idx:int) -> torch.tensor:
        buffer = self.tokens[idx*self.T:((idx+1)*self.T)+1]
        x = torch.tensor(buffer[:-1])
        y = torch.tensor(buffer[1:])

        return x, y

binary_file_path = '../data/tinystories/processed/train.bin'
T = -2 #49 #50 #98 #99 #100
tok_ds = TokenDataset_v2(binary_file_path, np.uint16, T, True)
print(tok_ds.tokens)
print('-'*100)
print(len(tok_ds))

[  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54
  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72
  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90
  91  92  93  94  95  96  97  98  99 100]
----------------------------------------------------------------------------------------------------
24


# DataLoader

In [79]:
# Dummy Dataset - V1
binary_file_path = '../data/tinystories/processed/train.bin'
T = -2
tok_ds = TokenDataset(binary_file_path, np.uint16, T, True)
print(tok_ds.tokens)
print('-'*100)
print('Length:', len(tok_ds))
print('-'*100)

# Dummy DataLoader
B=1
tok_dl_v1 = DataLoader(dataset=tok_ds, batch_size=B, drop_last=True)

for x, y in tok_dl_v1:
    print('x:\n', x)
    print('y:\n', y)
    print('-'*100)


[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48
 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72
 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96
 97]
----------------------------------------------------------------------------------------------------
Length: 24
----------------------------------------------------------------------------------------------------
x:
 tensor([[1, 2, 3, 4]])
y:
 tensor([[2, 3, 4, 5]])
----------------------------------------------------------------------------------------------------
x:
 tensor([[5, 6, 7, 8]])
y:
 tensor([[6, 7, 8, 9]])
----------------------------------------------------------------------------------------------------
x:
 tensor([[ 9, 10, 11, 12]])
y:
 tensor([[10, 11, 12, 13]])
----------------------------------------------------------------------------------------------------
x:
 tensor([[13, 

In [80]:
# Dummy Dataset - V2
binary_file_path = '../data/tinystories/processed/train.bin'
T = -2
tok_ds = TokenDataset_v2(binary_file_path, np.uint16, T, True)
print(tok_ds.tokens)
print('-'*100)
print('Length:', len(tok_ds))
print('-'*100)

# Dummy DataLoader
B=1
tok_dl_v2 = DataLoader(dataset=tok_ds, batch_size=B, drop_last=True)

for x, y in tok_dl_v2:
    print('x:\n', x)
    print('y:\n', y)
    print('-'*100)


[  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54
  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72
  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90
  91  92  93  94  95  96  97  98  99 100]
----------------------------------------------------------------------------------------------------
Length: 24
----------------------------------------------------------------------------------------------------
x:
 tensor([[1, 2, 3, 4]])
y:
 tensor([[2, 3, 4, 5]])
----------------------------------------------------------------------------------------------------
x:
 tensor([[5, 6, 7, 8]])
y:
 tensor([[6, 7, 8, 9]])
----------------------------------------------------------------------------------------------------
x:
 tensor([[ 9, 10, 11, 12]])
y:
 tensor([[10, 11, 12, 13]])
--------

# LLM Training Data: `np.memmap`, Dataset/DataLoader & Epochs

## 1. `np.memmap` — basics

`np.memmap` provides a **NumPy-array-like view over a file on disk** without loading the entire file into RAM.

```python
tokens = np.memmap(
    "train.bin",
    dtype=np.uint16,
    mode="r"
)
```

* `tokens` behaves much like a NumPy array: `tokens[i]`, `tokens[i:j]`, `len(tokens)`, etc.
* The file is **memory-mapped**; the OS loads the required disk pages into RAM as they are accessed.
* Therefore, a huge `train.bin` (e.g. several GB) does **not** require several GB of RAM just to create the `memmap`.
* `len(tokens)` does not scan the entire array. The number of elements can be determined from the file size and `dtype`, so this is essentially an O(1) metadata operation.

### Important distinction

```python
tokens = np.memmap(...)
```

does **not** load the data.

```python
x = tokens[1000:1128]
```

accesses only the relevant portion/pages.

```python
x = np.array(tokens[:])
```

materializes the entire mapped array as a normal NumPy array → potentially loads the whole dataset into RAM and defeats the main benefit of `memmap`.

---

## 2. Why `tokens = tokens[:]` is OK

A plain slice:

```python
tokens = tokens[:]
```

is essentially a **view of the same memory-mapped data**. It does not by itself create a full in-memory copy.

So this is still fine:

```python
tokens = np.memmap(...)
tokens = tokens[:]
```

The problematic operation is explicitly converting it to a normal NumPy array:

```python
tokens = np.array(tokens[:])
```

because now we are asking NumPy to materialize all values in RAM.

For the Dataset, therefore, it is best simply to avoid unnecessary `tokens[:]` altogether and leave the `memmap` untouched.

---

## 3. Epochs vs LLM token-based training

In conventional supervised learning:

```text
finite dataset of N (X, Y) samples
        ↓
one epoch = consume all N samples once
```

Epochs are therefore a natural unit of training.

LLM pretraining is naturally viewed as training over a **large stream of tokens**.

For example:

```text
batch_size = 32
context_length = 128

tokens / optimizer step = 32 × 128 = 4096
```

Therefore:

```text
10,000 optimizer steps
→ 40,960,000 training tokens
```

This is why LLM training is commonly described in terms of:

```text
"trained on 40M tokens"
```

rather than:

```text
"trained for 5 epochs"
```

An epoch still exists conceptually if the training corpus is finite — it simply isn't necessarily the most useful quantity for defining the training budget.

---

## 4. Dataset + DataLoader does NOT imply epoch-based training

A `DataLoader` is just an iterator over batches. It does not force the training loop to be written as:

```python
for epoch in range(num_epochs):
    for batch in train_loader:
        ...
```

If the DataLoader is exhausted, its iterator raises `StopIteration`. We can simply create a new iterator and continue.

A clean approach is an infinite loader:

```python
def infinite_loader(dataloader):
    while True:
        for batch in dataloader:
            yield batch
```

Then:

```python
train_iterator = infinite_loader(train_loader)

for step in range(max_steps):
    x, y = next(train_iterator)
    ...
```

Now the training loop is controlled by **steps/tokens**, not epochs.

For example:

```python
target_tokens = 5_000_000

tokens_per_step = batch_size * context_length

max_steps = target_tokens // tokens_per_step
```

The DataLoader may internally complete and restart its traversal several times, which is effectively multiple epochs, but **epochs never need to appear in the training logic**.

This breaks the false association:

```text
DataLoader → must use epochs
```

The actual relationship is:

```text
Dataset → defines samples
DataLoader → batches samples
Trainer → decides how long to train
```

---

## 5. Dataset/DataLoader vs custom LLM data loader

For this project, a standard `Dataset + DataLoader` setup is useful for learning PyTorch's data-loading abstractions.

However, for LLM pretraining specifically, a custom batch loader can be more natural:

```text
train.bin
    ↓
random / sequential starting positions
    ↓
[B, T] input tokens
[B, T] target tokens
```

This is close to the approach used in nanoGPT.

So there is nothing inherently wrong with either approach:

### Standard PyTorch approach

```text
memmap
  ↓
Dataset
  ↓
DataLoader
  ↓
Trainer
```

### LLM-oriented custom approach

```text
memmap
  ↓
custom get_batch()
  ↓
Trainer
```

For **LLM training**, the custom loader is arguably more natural because the underlying data is fundamentally a token stream rather than a conventional collection of independent `(X, Y)` samples.

The important thing is to understand what the abstractions are doing before deciding to remove them.

---

## 6. Non-overlapping LLM training samples

For context length `T`, non-overlapping samples can be defined as:

```text
sample 0:
X = tokens[0:T]
Y = tokens[1:T+1]

sample 1:
X = tokens[T:2T]
Y = tokens[T+1:2T+1]

sample 2:
X = tokens[2T:3T]
Y = tokens[2T+1:3T+1]
```

Thus consecutive `X` samples do not overlap.

The important distinction is:

```text
X and Y within one sample → shifted by 1 token
different samples         → non-overlapping
```

This avoids the heavy redundancy of stride-1 sliding windows.

For a token array of length `N`, the number of complete non-overlapping samples is:

```python
(N - 1) // T
```

The `-1` is required because the final input token needs a corresponding next-token target.

It is unnecessary to physically trim the `memmap`; simply make `__len__()` return the number of valid samples and leave unused trailing tokens untouched.


# Custom DataLoader

In [90]:
class TokenDataLoader:
    def __init__(self, B:int, T:int, binary_file_path:str, dtype:np.dtype, debug:bool=False):
        self.B = max(1, B)
        self.T = max(4, T)
        if not debug:
            self.tokens = np.memmap(
                filename=binary_file_path,
                dtype=dtype,
                mode='r'
            )
        else:
            self.tokens = np.arange(1, 101)
        if len(self.tokens)<=(self.B)*(self.T):
            raise ValueError(
                f"Current values of batch size {B} and seq length {T} are too large for dataset, please reduce either or both"
            )
        self.curr_idx = 0

    def next_batch(self) -> Tuple[torch.tensor]:
        B, T = (self.B), (self.T)

        buffer = self.tokens[self.curr_idx:self.curr_idx+(B*T+1)]
        x = torch.tensor(buffer[:-1]).view(B, T)
        y = torch.tensor(buffer[1:]).view(B, T)

        self.curr_idx += B*T
        if self.curr_idx+(B*T+1) > len(self.tokens):
            self.curr_idx=0

        return x, y


binary_file_path = '../data/tinystories/processed/train.bin'
B = -2
T = -49
tok_dl = TokenDataLoader(B, T, binary_file_path, np.uint16, True)
print(tok_dl.tokens)
print('-'*100)
steps = 10
for _ in range(steps):
    x, y = tok_dl.next_batch()
    print('x:\n', x)
    print('y:\n', y)    
print('-'*100)

[  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54
  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72
  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90
  91  92  93  94  95  96  97  98  99 100]
----------------------------------------------------------------------------------------------------
x:
 tensor([[1, 2, 3, 4]])
y:
 tensor([[2, 3, 4, 5]])
x:
 tensor([[5, 6, 7, 8]])
y:
 tensor([[6, 7, 8, 9]])
x:
 tensor([[ 9, 10, 11, 12]])
y:
 tensor([[10, 11, 12, 13]])
x:
 tensor([[13, 14, 15, 16]])
y:
 tensor([[14, 15, 16, 17]])
x:
 tensor([[17, 18, 19, 20]])
y:
 tensor([[18, 19, 20, 21]])
x:
 tensor([[21, 22, 23, 24]])
y:
 tensor([[22, 23, 24, 25]])
x:
 tensor([[25, 26, 27, 28]])
y:
 tensor([[26, 27, 28, 29]])
x:
 tensor([[29, 30, 31, 32]])
y:
 tensor([[30, 31, 32, 33]])
x:
 tensor([